# DE.LAS — Ciclo de vida de engenharia de dados (tutorial)

Este notebook roda no **Google Colab** e percorre quatro etapas típicas do dia a dia de uma pessoa engenheira de dados:

1. **Ingestão** — buscar dados de uma fonte externa (API pública).
2. **Armazenamento** — persistir dados brutos para reprodutibilidade e auditoria.
3. **Transformação** — modelar e enriquecer dados com **PySpark**.
4. **Disponibilização** — preparar uma camada analítica para consumo (consultas, dashboards, APIs internas).

Usamos a [PokeAPI](https://pokeapi.co/) como exemplo de fonte externa real, com JSON aninhado — cenário comum em integrações.

## Ambiente (Colab + PySpark)

Execute a célula abaixo uma vez por sessão. Ela instala dependências e cria a `SparkSession`.

In [ ]:
!pip install -q pyspark requests matplotlib pandas

In [ ]:
import os

from pyspark.sql import SparkSession

# Cria uma nova sessão de Spark para executar transformações de dados posteriormente
spark = (
    SparkSession.builder
    .appName("DE_LAS_Pokemon")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark

## 1. Ingestão de dados

Baixamos uma lista de Pokémon e, para cada um, o detalhe completo (tipos, estatísticas base, etc.). Em produção você usaria **retentativas**, **timeouts**, **paginação** e talvez uma **fila**; aqui mantemos o exemplo direto e educativo.

In [ ]:
import json
import time
from pathlib import Path

import requests

# Configurações para acessar a API do PokeAPI e coletar dados de Pokémon
BASE = "https://pokeapi.co/api/v2"
LIMIT = 40  # aumente com moderação para não sobrecarregar a API

# Cria uma sessão do módulo Python `requests`, que vai ser utilizada múltiplas requisições para a API do PokeAPI
session = requests.Session()
session.headers.update({"User-Agent": "DE.LAS-bootcamp-educational/1.0"})

# Faz uma requisição para a API do PokeAPI para obter uma lista de Pokémon, limitando o número de resultados
listing = session.get(f"{BASE}/pokemon", params={"limit": LIMIT}, timeout=60)
listing.raise_for_status()
results = listing.json()["results"]

# Para cada Pokémon listado, faz uma requisição para obter detalhes adicionais deles e armazena os resultados em uma lista Python
records = []
for item in results:
    detail = session.get(item["url"], timeout=60)
    detail.raise_for_status()
    records.append(detail.json())
    time.sleep(0.05)  # pequena pausa entre chamadas

# Exibe o total e o nome do primeiro Pokémon coletado
len(records), records[0]["name"]

### Note que...
- Estamos usando uma simples requisição web via módulo `requests` do Python, que é suficiente para a maioria dos casos de ingestão leve. Em cenários mais complexos, você pode precisar de ferramentas como **Apache Airflow**, **Luigi** ou **Prefect** para orquestrar pipelines de ingestão mais robustos e escaláveis.
- A etapa de ingestão **não deve ser acoplada à transformação**: aqui estamos trazendo os dados exatamente como a API retorna, sem nenhuma modificação. Isso é importante para garantir a reprodutibilidade e facilitar auditorias futuras. Se quisermos transformar os dados, faremos isso na etapa de transformação, usando o Spark.

## 2. Armazenamento (camada bruta / bronze)

Gravamos os JSON completos em **JSON Lines** (um documento JSON por linha). Esse formato é fácil de versionar e de reler com Spark, sem perder o detalhe original.

In [ ]:
# Define o local de salvamento dos dados
DATA_DIR = Path("./content/delas_data")
RAW_PATH = DATA_DIR / "bronze" / "pokemon_raw.jsonl"
RAW_PATH.parent.mkdir(parents=True, exist_ok=True)

# Para cada registro de Pokémon coletado, salva os dados em um arquivo JSONL, onde cada linha é um objeto JSON representando um Pokémon completo
with RAW_PATH.open("w", encoding="utf-8") as f:
    for row in records:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Exibe o caminho do arquivo salvo e o número de linhas (Pokémon) que foram armazenados
RAW_PATH, sum(1 for _ in RAW_PATH.open())

In [ ]:
# Exibe apenas a primeira linha do arquivo JSONL para verificar o formato dos dados salvos
!head -n 1 {RAW_PATH}

### Note que...
- Estamos salvando os dados brutos exatamente como foram ingeridos, sem nenhuma modificação. Se quisermos transformar os dados, faremos isso na etapa de transformação, usando o Spark.
- Em um ambiente de produção, você provavelmente usaria um sistema de arquivos distribuído como **Google Cloud Storage** ou **Amazon S3** para armazenar os dados brutos, garantindo escalabilidade e durabilidade.

## 3. Transformação (PySpark)

Lemos o bronze com Spark, **explodimos** arrays aninhados (tipos e stats) e calculamos colunas úteis para análise.

In [ ]:
from pyspark.sql import functions as F

# Lê os dados brutos do arquivo JSONL usando o Spark, criando um DataFrame que pode ser manipulado
raw_df = spark.read.json(str(RAW_PATH))

# Transforma os dados para criar um DataFrame mais estruturado, onde cada tipo de Pokémon é representado em uma linha separada
types_df = (
    raw_df
    .select("id", "name", F.explode("types").alias("t"))
    .select("id", "name", F.col("t.type.name").alias("type_name"), F.col("t.slot").alias("type_slot"))
)

# Transforma os dados para criar um DataFrame mais estruturado, onde cada estatística de Pokémon é representada em uma linha separada
stats_df = (
    raw_df
    .select("id", "name", F.explode("stats").alias("s"))
    .select(
        "id",
        "name",
        F.col("s.stat.name").alias("stat_name"),
        F.col("s.base_stat").alias("base_stat"),
    )
)

# Faz uma operação de "pivot" para transformar as linhas de estatísticas em colunas, onde cada tipo de estatística se torna uma coluna separada com o valor correspondente
pivot_stats = (
    stats_df
    .groupBy("id", "name")
    .pivot("stat_name")
    .agg(F.first("base_stat"))
)

# Faz a junção dos DataFrames de tipos e estatísticas para criar um DataFrame final que contém todas as informações relevantes sobre cada Pokémon, incluindo seus tipos e estatísticas
curated = types_df.join(pivot_stats, on=["id", "name"], how="inner")

# Exibe o esquema do DataFrame final e as primeiras linhas para verificar a estrutura dos dados transformados
curated.printSchema()
curated.show(5, truncate=False)

In [ ]:
types_df.show(5)

### Note que...
- Estamos manipulando os dados usando o Spark, que é uma ferramenta poderosa para transformação de grandes volumes de dados.
- A etapa de transformação é onde aplicamos a lógica de negócio para modelar os dados de forma que sejam mais fáceis de consumir posteriormente. Aqui, por exemplo, estamos transformando os tipos e estatísticas dos Pokémon em um **formato tabular mais tradicional**, o que facilita consultas e análises futuras. Em um ambiente de produção, você pode querer adicionar etapas adicionais de **limpeza**, **validação** e **enriquecimento dos dados** durante a transformação, dependendo dos requisitos do seu caso de uso.
- Outras ferramentas de transformação, como **dbt** ou **pandas**, também podem ser usadas dependendo do contexto e do volume de dados.

## 4. Disponibilização (camada ouro)

Materializamos uma tabela agregada (contagem de Pokémon por tipo primário) em **Parquet** — formato colunar eficiente para inteligência de negócio e treino de modelos. Também registramos uma **visão temporária** para consultas SQL pontuais e geramos uma **visualização** simples a partir da tabela ouro (como faria um painel ou um notebook de negócio).

In [ ]:
# Filtra o DataFrame de tipos para obter apenas o tipo primário de cada Pokémon (onde `type_slot` é igual a 1) e renomeia a coluna do tipo para "primary_type"
primary_type = types_df.filter(F.col("type_slot") == 1).select("id", F.col("type_name").alias("primary_type"))

# Agrupa os dados por tipo primário e conta o número de Pokémon distintos em cada tipo, ordenando os resultados em ordem decrescente de contagem
by_type = (
    primary_type
    .groupBy("primary_type")
    .agg(F.countDistinct("id").alias("pokemon_count"))
    .orderBy(F.desc("pokemon_count"))
)

# Exibe o resultado da contagem de Pokémon por tipo primário, mostrando o tipo e a quantidade de Pokémon associados a ele
GOLD_DIR = DATA_DIR / "gold" / "pokemon_by_primary_type"
GOLD_DIR.parent.mkdir(parents=True, exist_ok=True)

# Salva o DataFrame resultante da contagem de Pokémon por tipo primário em um formato Parquet, sobrescrevendo qualquer arquivo existente no local de destino
by_type.write.mode("overwrite").parquet(str(GOLD_DIR))

# Cria uma visão temporária no Spark SQL para permitir consultas SQL sobre o DataFrame de contagem de Pokémon por tipo primário
by_type.createOrReplaceTempView("pokemon_by_primary_type")

# Executa uma consulta SQL para selecionar o tipo primário e a contagem de Pokémon, ordenando os resultados em ordem decrescente de contagem, e exibe os resultados
spark.sql("""
  SELECT primary_type, pokemon_count
  FROM pokemon_by_primary_type
  ORDER BY pokemon_count DESC
""").show(truncate=False)

In [ ]:
import matplotlib.pyplot as plt

# Converte o DataFrame do Spark para um DataFrame do Pandas para facilitar a visualização usando Matplotlib
pdf = by_type.toPandas()

# Cria um gráfico de barras para visualizar a distribuição de Pokémon por tipo primário, configurando os rótulos dos eixos, o título do gráfico e a rotação dos rótulos do eixo x para melhor legibilidade
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(pdf["primary_type"], pdf["pokemon_count"], color="#2E86AB")
ax.set_xlabel("Tipo primário")
ax.set_ylabel("Pokémon distintos (contagem)")
ax.set_title("Distribuição por tipo primário (amostra)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Note que...
- Na camada ouro, estamos preparando os dados para **consumo final**, seja por analistas, cientistas de dados ou aplicações. Em geral, aplicamos transformações finais, como **agregações**, **cálculos de métricas** e **modelagem de dados**, para criar uma estrutura que seja fácil de consultar e interpretar.
- A criação de uma **visão temporária no Spark SQL** é uma prática comum para permitir consultas via queries SQL e análises exploratórias sem a necessidade de criar tabelas permanentes. Em um ambiente de produção, você pode querer criar tabelas permanentes ou usar um catálogo de metadados para gerenciar suas tabelas e visões de forma mais estruturada (pense no benefício que um catálogo de dados traz para uma empresa, onde é possível entender todas as tabelas que a empresa possui).
- O formato Parquet é escolhido por sua eficiência em leitura e compactação, o que é ideal para análises rápidas e escaláveis.

### Encerramento

- Você percorreu **ingestão → armazenamento → transformação → disponibilização** com uma API real e PySpark, incluindo uma visualização da camada ouro.
- Próximo passo: abra **`lab1_berry_pipeline_exercise.ipynb`** (rota da API e agregações diferentes das do tutorial) e complete os `# TODO`. O gabarito está em **`lab1_berry_pipeline_exercise_solutions.ipynb`**.